In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# =========================================================================
# 1. CARGA DEL DATASET PROPIO (Cambia el nombre si usas dataset_real_garagoa.csv)
# =========================================================================
archivo = "dataset_ladera_v1.csv"
df = pd.read_csv(archivo)

# Ajustamos nombres de columna automáticamente por si usas v1 o el real
col_angulo = "angulo_limpio" if "angulo_limpio" in df.columns else "angulo_pitch"
X = df[["humedad_suelo_%", col_angulo, "velocidad_inclinacion"]].values.astype(np.float32)
y = df["etiqueta"].values.astype(np.int32)

# Normalización Z-Score (guardamos media y desviación para ponerlas en el ESP32)
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0) + 1e-7
X_norm = (X - X_mean) / X_std

# Particionamiento estratificado: 70% Train, 15% Val, 15% Test (Requisito R4)
X_train, X_temp, y_train, y_temp = train_test_split(X_norm, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Muestras -> Entrenamiento: {len(X_train)} | Validación: {len(X_val)} | Prueba: {len(X_test)}")

# =========================================================================
# 2. ENTRENAMIENTO DEL MODELO LIGERO (MLP)
# =========================================================================
model = models.Sequential([
    layers.Input(shape=(3,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8, activation='relu'),
    layers.Dense(3, activation='softmax') # 3 clases: 0=Estable, 1=Riesgo, 2=Deslizamiento
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=30, batch_size=16, validation_data=(X_val, y_val), verbose=0)

# Evaluación del modelo original (Float32)
pred_float = np.argmax(model.predict(X_test, verbose=0), axis=1)
acc_float = accuracy_score(y_test, pred_float) * 100
f1_float = f1_score(y_test, pred_float, average='weighted') * 100

# =========================================================================
# 3. CUANTIZACIÓN A TENSORFLOW LITE (INT8) PARA EL ESP32 (Requisito R3)
# =========================================================================
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_data_gen():
    for muestra in X_train[:100]:
        yield [np.array([muestra], dtype=np.float32)]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

# Evaluar el modelo cuantizado INT8 para medir pérdida/ganancia (Requisito R3)
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

scale, zero_point = input_details['quantization']
pred_int8 = []
for muestra in X_test:
    entrada_q = np.array([muestra / scale + zero_point], dtype=np.int8)
    interpreter.set_tensor(input_details['index'], entrada_q)
    interpreter.invoke()
    salida_q = interpreter.get_tensor(output_details['index'])
    pred_int8.append(np.argmax(salida_q))

acc_int8 = accuracy_score(y_test, pred_int8) * 100
f1_int8 = f1_score(y_test, pred_int8, average='weighted') * 100

print("\n=== TABLA DE MÉTRICAS PARA EL INFORME IEEE (REQUISITO R3) ===")
print(f"Modelo Original (Float32) -> Accuracy: {acc_float:.2f}% | F1-Score: {f1_float:.2f}%")
print(f"Modelo Cuantizado (INT8)  -> Accuracy: {acc_int8:.2f}% | F1-Score: {f1_int8:.2f}%")
print(f"Tamaño final del modelo   -> {len(tflite_model)} bytes ({len(tflite_model)/1024:.2f} KB)")

# =========================================================================
# 4. EXPORTAR ARCHIVO C++ (model.h) PARA DIEGO
# =========================================================================
hex_lines = [f'0x{b:02x}' for b in tflite_model]
c_code = "// Archivo generado automáticamente para el ESP32 - Proyecto Garagoa\n"
c_code += "const unsigned char g_model[] = {\n  "
c_code += ",\n  ".join(", ".join(hex_lines[i:i+12]) for i in range(0, len(hex_lines), 12))
c_code += "\n};\n"
c_code += f"const int g_model_len = {len(tflite_model)};\n"
c_code += f"const float X_MEAN[3] = {{{X_mean[0]:.4f}f, {X_mean[1]:.4f}f, {X_mean[2]:.4f}f}};\n"
c_code += f"const float X_STD[3]  = {{{X_std[0]:.4f}f, {X_std[1]:.4f}f, {X_std[2]:.4f}f}};\n"

with open("model.h", "w") as f:
    f.write(c_code)

print("\n¡Éxito! Descarga el archivo 'model.h' de la carpeta izquierda y pásaselo a Diego.")

Muestras -> Entrenamiento: 350 | Validación: 75 | Prueba: 75
Saved artifact at '/tmp/tmp2clhdxes'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133237625352080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133237618788496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133237618788112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133237618788880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133240121275984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133237618788688: TensorSpec(shape=(), dtype=tf.resource, name=None)

=== TABLA DE MÉTRICAS PARA EL INFORME IEEE (REQUISITO R3) ===
Modelo Original (Float32) -> Accuracy: 94.67% | F1-Score: 94.72%
Modelo Cuantizado (INT8)  -> Accuracy: 94.67% | F1-Score: 94.72%
Tamaño final del modelo   -> 3472 bytes (3.39 KB)

¡

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
